Look for all products contained in a folder and overlay their footprints on a map

In [ ]:
%load_ext autoreload
%autoreload 2
from zipfile import ZipFile, is_zipfile
from glob import glob
import xmltodict
from folium import Map, GeoJson, LayerControl, Tooltip
import numpy as np
import geopandas as gpd
from shapely import Polygon
import os.path

In [ ]:
m = Map()
data_dir = "/data/S1"
files = glob(f"{data_dir}/S1*.zip")
for file in files:
    product = os.path.splitext(os.path.basename(file))[0]
    z = ZipFile(file)

    kml_file = [it for it in z.namelist() if it.endswith("kml")][0]
    kml_dict = xmltodict.parse(z.read(kml_file))["kml"]
    kml_geom = kml_dict["Document"]["Folder"]["GroundOverlay"]["gx:LatLonQuad"]
    new_dict = {"id": product, "type": "Polygon", "coordinates": []}
    for it in kml_geom['coordinates'].split(" "):
        ll = list(map(float, it.split(",")))
        new_dict["coordinates"].append(ll)
    center = np.array(new_dict["coordinates"]).mean(0)
    gdf = gpd.GeoDataFrame(geometry=[Polygon(new_dict["coordinates"])])
    gdf.crs="EPSG:4326"
    Tooltip(product).add_to(GeoJson(gdf, name=product).add_to(m))
LayerControl().add_to(m)
m